# BLIP 图像描述生成与视觉问答

这个 Notebook 展示 `BLIP（Bootstrapping Language-Image Pre-training）` 做图像描述生成（Image Captioning）和 VQA 的完整流程。

内容包括：
- BLIP 架构解读：Image Encoder + Text Encoder/Decoder 双流 + Bootstrap 自引导策略
- 与 CLIP（对齐表示）和 ViLT（单塔推理）的三角对比
- 无条件 / 条件图像描述生成
- Visual Question Answering 演示
- 多图批量描述展示

## 1. 环境准备

```bash
pip install torch transformers pillow requests matplotlib
```

In [ ]:
from dataclasses import dataclass
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipForQuestionAnswering, BlipProcessor

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    caption_model: str = 'Salesforce/blip-image-captioning-base'
    vqa_model: str    = 'Salesforce/blip-vqa-base'
    max_new_tokens: int = 50
    num_beams: int = 4

cfg = Config()
cfg

## 2. 加载示例图像

In [ ]:
def load_image(url):
    resp = requests.get(url, timeout=10)
    return Image.open(BytesIO(resp.content)).convert('RGB')


image_data = [
    ('猫咪',  'http://images.cocodataset.org/val2017/000000039769.jpg'),
    ('厨房',  'http://images.cocodataset.org/val2017/000000283900.jpg'),
    ('运动',  'http://images.cocodataset.org/val2017/000000001761.jpg'),
    ('街景',  'http://images.cocodataset.org/val2017/000000397133.jpg'),
]

images = [(label, load_image(url)) for label, url in image_data]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (label, img) in zip(axes, images):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. BLIP Processor 与模型加载

In [ ]:
caption_processor = BlipProcessor.from_pretrained(cfg.caption_model)
caption_model = BlipForConditionalGeneration.from_pretrained(cfg.caption_model).to(device)
caption_model.eval()
print('Caption 模型加载完成')

## 4. BLIP 结构解读

### 4.1 与 CLIP、ViLT 的三角对比

| 维度 | CLIP | ViLT | BLIP |
|------|------|------|------|
| 目标 | 图文对齐（对比学习） | 图文融合推理（VQA） | 图文理解 + 生成 |
| 架构 | 双塔，无交叉注意力 | 单塔，早期融合 | 双流：Image Encoder + Text Encoder/Decoder |
| 生成能力 | 无（只做对齐） | 无（only 分类） | **有**（自回归生成描述） |
| 预训练数据 | 4 亿图文对（含噪声） | 图文对 | 图文对 + **噪声过滤（Bootstrap）** |
| 适合任务 | 检索、zero-shot 分类 | VQA、图文推理 | 描述生成、VQA、检索 |

### 4.2 Bootstrap 自引导策略

BLIP 的核心创新：网络爬取的图文对噪声严重，BLIP 用一个 **Captioner 模型** 为图像生成新的描述，再用 **Filter 模型** 过滤与图像不符的描述，从而提升训练数据质量。这个循环自举（Bootstrap）过程使 BLIP 在有限标注数据下仍能学到高质量表示。

### 4.3 Text Decoder（自回归生成）

与 ViLT 只能做 closed-set 分类不同，BLIP 的 Text Decoder 是真正的自回归语言模型，能生成任意长度的开放式描述。

## 5. 无条件图像描述生成

In [ ]:
@torch.no_grad()
def generate_caption(model, processor, image, cfg, condition=None):
    if condition:
        inputs = processor(image, condition, return_tensors='pt').to(device)
    else:
        inputs = processor(image, return_tensors='pt').to(device)

    # beam search 比 greedy 生成质量更高，但慢一些
    ids = model.generate(**inputs, max_new_tokens=cfg.max_new_tokens, num_beams=cfg.num_beams)
    return processor.decode(ids[0], skip_special_tokens=True)


print('=== 无条件描述生成 ===')
for label, img in images:
    caption = generate_caption(caption_model, caption_processor, img, cfg)
    print(f'[{label}] {caption}')

## 6. 条件图像描述生成

给模型一个文本前缀作为条件，引导生成特定风格或角度的描述。

In [ ]:
_, cat_image = images[0]

conditions = [
    None,
    'a photo of',
    'there are two',
    'the cats are',
]

print('=== 条件描述生成（猫咪图）===')
for cond in conditions:
    cap = generate_caption(caption_model, caption_processor, cat_image, cfg, condition=cond)
    prefix = f'前缀=\"{cond}\"' if cond else '无条件'
    print(f'{prefix:20s} -> {cap}')

## 7. Visual Question Answering 演示

In [ ]:
vqa_processor = BlipProcessor.from_pretrained(cfg.vqa_model)
vqa_model = BlipForQuestionAnswering.from_pretrained(cfg.vqa_model).to(device)
vqa_model.eval()


@torch.no_grad()
def blip_vqa(model, processor, image, question, device):
    inputs = processor(image, question, return_tensors='pt').to(device)
    ids = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(ids[0], skip_special_tokens=True)


qa_pairs = [
    (images[0][1], 'How many cats are in the image?'),
    (images[0][1], 'What are the cats doing?'),
    (images[1][1], 'What room is this?'),
    (images[1][1], 'Is there a stove in the kitchen?'),
]

print('=== BLIP VQA（开放式生成）===')
for img, q in qa_pairs:
    ans = blip_vqa(vqa_model, vqa_processor, img, q, device)
    print(f'Q: {q}')
    print(f'A: {ans}\n')

## 8. 多图批量描述展示

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (label, img) in zip(axes.flatten(), images):
    caption = generate_caption(caption_model, caption_processor, img, cfg)
    ax.imshow(img)
    # 长描述自动换行
    import textwrap
    wrapped = '\n'.join(textwrap.wrap(caption, width=50))
    ax.set_title(f'[{label}]\n{wrapped}', fontsize=9)
    ax.axis('off')

plt.suptitle('BLIP 图像描述生成结果', y=1.01)
plt.tight_layout()
plt.show()